# Extraccion de entidades ENFERMEDAD con Qwen2.5-32B

Cuaderno para cargar el modelo en 4-bit, procesar hasta 250 textos de Distemist y evaluar en mencion estricta.

In [1]:
%pip install -q transformers accelerate torch huggingface_hub pandas hf_transfer bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Configuracion

Se definen modelo, rutas y numero de textos a procesar.

In [2]:
import json
import re
import time
from pathlib import Path
import warnings
import os

import gc
import pandas as pd
import torch
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from transformers import logging

warnings.filterwarnings("ignore")
logging.set_verbosity_error()

MODEL_ID = "unsloth/Qwen2.5-32B-Instruct-bnb-4bit"
N_TEXTS = 250
MAX_NEW_TOKENS = 512

PROJECT_ROOT = "/kaggle/input/datasets/carloshonrado"
DISTEMIST_ROOT = f"{PROJECT_ROOT}/distemist/distemist"
TEXT_FILES_DIR = f"{DISTEMIST_ROOT}/text_files"

DATA_PATHS = {
    "train_jsonl": f"{DISTEMIST_ROOT}/distemist_train.jsonl",
    "test_jsonl": f"{DISTEMIST_ROOT}/distemist_test.jsonl",
    "text_files_dir": TEXT_FILES_DIR,
    "gs_mentions_tsv": f"{DISTEMIST_ROOT}/distemist_subtrack1_test_mentions.tsv",
}

PREDICTIONS_TSV = "distemist_llm_predictions.tsv"
EVAL_SUMMARY_JSON = "evaluation_summary.json"

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
text_dir = Path(DATA_PATHS["text_files_dir"])
MAX_TEXTS = len(list(text_dir.glob("*.txt")))
txt_files = sorted(text_dir.glob("*.txt"))[: min(N_TEXTS, MAX_TEXTS)]

print(f"torch={torch.__version__} | cuda={torch.cuda.is_available()}")
print(f"MAX_TEXTS disponibles: {MAX_TEXTS}")
print(f"Textos a procesar: {len(txt_files)}")
print(f"GS: {DATA_PATHS['gs_mentions_tsv']}")

torch=2.10.0+cu128 | cuda=True
MAX_TEXTS disponibles: 250
Textos a procesar: 250
GS: /kaggle/input/datasets/carloshonrado/distemist/distemist/distemist_subtrack1_test_mentions.tsv


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
print("Tokenizer cargado")

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", token=hf_token)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)
print("Modelo cargado")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modelo cargado


In [4]:
def build_ner_messages(text):
    return [
        {
            "role": "system",
            "content": (
                "Actúa como un sistema NER médico de alta precisión.\n\n"
                "REGLAS DE EXTRACCIÓN:\n"
                "1. Extrae exclusivamente entidades de tipo ENFERMEDAD.\n"
                "2. COPIA Y PEGA de forma literal: No cambies mayúsculas, minúsculas ni tildes.\n"
                "3. PROHIBIDO USAR SINÓNIMOS: Si el texto dice 'neoplasia', no escribas 'cáncer'.\n"
                "4. REPETICIONES: Si una enfermedad aparece 3 veces en el texto, debes listarla 3 veces en líneas separadas.\n"
                "5. ORDEN: Extrae las menciones en el mismo orden en que aparecen en el texto.\n"
                "6. FORMATO: Solo la mención, una por línea. Sin prefijos, guiones, viñetas ni comentarios.\n"
                "7. Si no hay nada, devuelve un texto vacío."
            ),
        },
        {
            "role": "user",
            "content": f"Texto para analizar:\n{text}",
        },
    ]

## Extraccion

Se genera la salida del modelo y se construye el TSV de predicciones con offsets.

In [5]:
pred_rows = []
inference_times = []
total_start = time.time()
total_files = len(txt_files)

for i, txt_path in enumerate(txt_files):
    text = txt_path.read_text(encoding="utf-8")
    prompt = tokenizer.apply_chat_template(
        build_ner_messages(text),
        tokenize=False,
        add_generation_prompt=True,
    )

    try:
        t0 = time.time()
        result = pipe(
            prompt,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )[0]["generated_text"].strip()
        
        t_infer = time.time() - t0
        inference_times.append(t_infer)

        mentions = [line.strip() for line in result.splitlines() if line.strip()]
        search_start = 0
        mark_counter = 1

        for mention in mentions:
            pattern = re.escape(mention)
            match = re.search(pattern, text[search_start:], re.IGNORECASE)

            if match:
                off0 = search_start + match.start()
                off1 = search_start + match.end()
                search_start = off1
            else:
                rescue = re.search(pattern, text, re.IGNORECASE)
                if not rescue:
                    continue
                off0, off1 = rescue.start(), rescue.end()

            pred_rows.append(
                {
                    "filename": txt_path.stem,
                    "mark": f"T{mark_counter}",
                    "label": "ENFERMEDAD",
                    "off0": off0,
                    "off1": off1,
                    "span": text[off0:off1],
                }
            )
            mark_counter += 1

        print(f"[{i + 1}/{total_files}] {txt_path.name} | {t_infer:.2f} s | {len(mentions)} menciones, {mark_counter - 1} localizadas")

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"[{i + 1}/{total_files}] {txt_path.name} | OOM ERROR ")
            gc.collect()
            torch.cuda.empty_cache()
        else:
            print(f"[{i + 1}/{total_files}] {txt_path.name} | ERROR: {e}")
        

pd.DataFrame(
    pred_rows,
    columns=["filename", "mark", "label", "off0", "off1", "span"],
).to_csv(PREDICTIONS_TSV, sep="\t", index=False)

total_time = time.time() - total_start
avg_time = sum(inference_times) / len(inference_times) if inference_times else 0.0

print(f"\nTiempo total: {total_time:.2f} s")
print(f"Tiempo medio por texto: {avg_time:.2f} s")
print(f"Predicciones guardadas en: {PREDICTIONS_TSV}")

[1/250] S0004-06142006000100010-1.txt | 25.04 s | 4 menciones, 4 localizadas
[2/250] S0004-06142006000100014-1.txt | 55.26 s | 9 menciones, 9 localizadas
[3/250] S0004-06142006000500002-2.txt | 29.76 s | 2 menciones, 2 localizadas
[4/250] S0004-06142006000500002-3.txt | 61.64 s | 10 menciones, 10 localizadas
[5/250] S0004-06142006000900015-1.txt | 37.90 s | 5 menciones, 5 localizadas
[6/250] S0004-06142007000200017-1.txt | 50.64 s | 5 menciones, 5 localizadas
[7/250] S0004-06142007000600012-1.txt | 47.29 s | 2 menciones, 2 localizadas
[8/250] S0004-06142007000600014-1.txt | 29.05 s | 2 menciones, 1 localizadas
[9/250] es-S0004-06142007000900010-1.txt | 29.59 s | 0 menciones, 0 localizadas
[10/250] es-S0004-06142007000900012-1.txt | 37.04 s | 8 menciones, 8 localizadas
[11/250] es-S0004-06142008000300011-1.txt | 37.41 s | 7 menciones, 7 localizadas
[12/250] es-S0004-06142008000300012-1.txt | 66.93 s | 6 menciones, 6 localizadas
[13/250] es-S0004-06142008000300013-1.txt | 50.22 s | 16 me

## Evaluacion

Se comparan predicciones y gold standard con criterio estricto de offsets.

In [6]:
df_pred = pd.read_csv(PREDICTIONS_TSV, sep="\t")
df_gs_all = pd.read_csv(DATA_PATHS["gs_mentions_tsv"], sep="\t")

processed_files = df_pred["filename"].unique()
df_gs = df_gs_all[df_gs_all["filename"].isin(processed_files)]

set_gs = set(zip(df_gs["filename"], df_gs["label"], df_gs["off0"], df_gs["off1"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["off0"], df_pred["off1"]))

tp = len(set_gs & set_pred)
fp = len(set_pred - set_gs)
fn = len(set_gs - set_pred)

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
fscore = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

report = {
    "model_id": MODEL_ID,
    "n_texts_requested": int(N_TEXTS),
    "n_texts_processed": int(len(processed_files)),
    "tp": int(tp),
    "fp": int(fp),
    "fn": int(fn),
    "precision": round(float(precision), 4),
    "recall": round(float(recall), 4),
    "fscore": round(float(fscore), 4),
    "predictions_file": PREDICTIONS_TSV,
}

with open(EVAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Archivos evaluados: {len(processed_files)}")
print(f"TP={tp} | FP={fp} | FN={fn}")
print(f"Precision={precision:.4f}")
print(f"Recall={recall:.4f}")
print(f"F1={fscore:.4f}")
print(f"Resumen guardado en: {EVAL_SUMMARY_JSON}")

Archivos evaluados: 238
TP=659 | FP=366 | FN=1865
Precision=0.6429
Recall=0.2611
F1=0.3714
Resumen guardado en: evaluation_summary.json
